[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week4_deep_learning/day28_transfer_learning/day28_notebook.ipynb)

# Day 28 / 42: Transfer Learning
### #42DaysOfML | Week 4: Deep Learning

---

## What You'll Learn
- What transfer learning is and why it works
- Feature extraction vs fine-tuning — when to use each
- Fine-tune MobileNetV2 on a custom 5-class dataset
- Compare accuracy: training from scratch vs transfer learning
- Layer freezing strategy and how to unfreeze progressively
- Production problem: domain shift in transfer learning

---

In [ ]:
!pip install torch torchvision matplotlib numpy --quiet

## The Concept

Training a CNN from scratch requires millions of labelled images and days of GPU compute. Most real-world tasks don't have that. A 3-person startup building a plant disease classifier has maybe 5,000 images and one GPU.

Transfer learning solves this by reusing knowledge from a model already trained on a large dataset. The key insight: a model trained on 1.2 million images (ImageNet) has already learned to detect edges, textures, shapes, and object parts. These low-level features are useful for almost any visual task. You don't need to learn them again.

**Two strategies:**

**1. Feature Extraction:** Freeze all pretrained layers. Only train the final classification head on your new data. Fast. Works well when your dataset is small (<1,000 images) and similar to ImageNet.

**2. Fine-tuning:** Unfreeze some or all pretrained layers and train them at a very low learning rate alongside your new head. Slower. Works better when you have more data or your domain is different from ImageNet (medical images, satellite imagery, etc.).

**Why it works mathematically:**
The pretrained weights are a better starting point than random initialisation. Gradient descent from that starting point converges faster and to a better solution — especially when you have limited data, where random initialisation would overfit immediately.

**How companies actually use this:**
- A 3-person startup building a medical image classifier used ResNet50 pretrained on ImageNet. With 2,000 labelled X-rays they reached 91% accuracy. Training from scratch with 2,000 images got 63%.
- Google's production image understanding pipelines are almost entirely based on pretrained Vision Transformers and EfficientNets, fine-tuned on task-specific data.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split, Subset
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# ============================================================
# SECTION 1: Simulate a Small Real-World Dataset
# We take 5 classes from CIFAR-10 with only 200 images each
# to simulate a real limited-data scenario.
# Total: 1,000 training images (vs 50,000 in full CIFAR-10)
# ============================================================

CLASSES_SELECTED = [0, 1, 2, 3, 4]  # plane, car, bird, cat, deer
CLASS_NAMES = ['plane', 'car', 'bird', 'cat', 'deer']
SAMPLES_PER_CLASS_TRAIN = 200  # Very small — transfer learning shines here
SAMPLES_PER_CLASS_TEST = 100

# MobileNetV2 expects 224x224 images (it was pretrained on ImageNet at this resolution)
# CIFAR-10 is 32x32, so we resize it
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    # ImageNet normalisation — must match what MobileNetV2 was pretrained on
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
full_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

def get_class_indices(dataset, class_ids, n_per_class):
    """Get n_per_class indices for each class_id from a dataset."""
    indices = []
    counts = {c: 0 for c in class_ids}
    for idx, (_, label) in enumerate(dataset):
        if label in class_ids and counts[label] < n_per_class:
            indices.append(idx)
            counts[label] += 1
        if all(v >= n_per_class for v in counts.values()):
            break
    return indices

train_indices = get_class_indices(full_train, CLASSES_SELECTED, SAMPLES_PER_CLASS_TRAIN)
test_indices = get_class_indices(full_test, CLASSES_SELECTED, SAMPLES_PER_CLASS_TEST)

small_train = Subset(full_train, train_indices)
small_test = Subset(full_test, test_indices)

# Remap labels to 0-4
class DatasetRemap(torch.utils.data.Dataset):
    def __init__(self, subset, class_map):
        self.subset = subset
        self.class_map = class_map
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return img, self.class_map[label]

class_map = {c: i for i, c in enumerate(CLASSES_SELECTED)}
train_dataset = DatasetRemap(small_train, class_map)
test_dataset = DatasetRemap(small_test, class_map)

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
testloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Training samples:   {len(train_dataset)} ({SAMPLES_PER_CLASS_TRAIN} per class)")
print(f"Test samples:       {len(test_dataset)} ({SAMPLES_PER_CLASS_TEST} per class)")
print(f"Classes:            {CLASS_NAMES}")
print(f"\nThis is intentionally a small dataset to show why transfer learning matters.")

In [ ]:
# ============================================================
# SECTION 2: Train From Scratch (Baseline)
# Use same small dataset — no pretrained weights
# ============================================================

class SimpleCNN(nn.Module):
    """Simple 3-block CNN, same architecture as Day 27 but for 5 classes."""
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, trainloader, testloader, epochs, lr, label):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_accs, val_accs = [], []
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            _, pred = model(inputs).max(1)
            correct += pred.eq(targets).sum().item()
            total += targets.size(0)
        train_accs.append(100. * correct / total)
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.to(device)
                _, pred = model(inputs).max(1)
                correct += pred.eq(targets).sum().item()
                total += targets.size(0)
        val_accs.append(100. * correct / total)
        scheduler.step()
        
        if (epoch + 1) % 5 == 0:
            print(f"[{label}] Epoch {epoch+1:3d}/{epochs}: Train {train_accs[-1]:.1f}% | Val {val_accs[-1]:.1f}%")
    
    elapsed = time.time() - start
    print(f"[{label}] Final Val Accuracy: {val_accs[-1]:.1f}% | Time: {elapsed:.1f}s")
    return train_accs, val_accs


print("Training from scratch on 1,000 images...")
scratch_model = SimpleCNN(num_classes=5)
scratch_train_accs, scratch_val_accs = train_model(scratch_model, trainloader, testloader, epochs=20, lr=0.001, label="Scratch")

In [ ]:
# ============================================================
# SECTION 3: Transfer Learning — Feature Extraction
# Freeze all pretrained layers, only train the classifier head
# ============================================================

def build_mobilenetv2_feature_extractor(num_classes=5):
    """
    MobileNetV2 pretrained on ImageNet.
    Strategy: freeze everything, replace + train only the final classifier.
    """
    model = models.mobilenet_v2(pretrained=True)
    
    # Freeze ALL pretrained layers
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace the classifier head (1280 -> num_classes)
    # Only these layers have requires_grad=True
    model.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(1280, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    
    # Count trainable vs frozen
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params:     {total:,}")
    print(f"Trainable params: {trainable:,} ({100*trainable/total:.1f}% of total)")
    print(f"Frozen params:    {total-trainable:,} ({100*(total-trainable)/total:.1f}% of total)")
    
    return model

print("Building MobileNetV2 (feature extraction mode)...")
fe_model = build_mobilenetv2_feature_extractor(num_classes=5)

# Only pass trainable params to optimizer
print("\nTraining feature extractor (only head) on 1,000 images...")
fe_train_accs, fe_val_accs = train_model(fe_model, trainloader, testloader, epochs=20, lr=0.001, label="Feature Extraction")

In [ ]:
# ============================================================
# SECTION 4: Transfer Learning — Fine-tuning
# Unfreeze last few blocks at a very low LR
# ============================================================

def build_mobilenetv2_finetune(num_classes=5):
    """
    MobileNetV2 with fine-tuning strategy:
    - Freeze early layers (learn generic features, don't change them)
    - Unfreeze last 3 inverted residual blocks
    - Replace and train classifier head
    
    Key: use a much lower LR for pretrained layers vs the head.
    """
    model = models.mobilenet_v2(pretrained=True)
    
    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False
    
    # Unfreeze last 3 feature blocks (blocks 15, 16, 17 out of 18)
    # These learn high-level features most relevant to your specific task
    unfreeze_from = len(model.features) - 3  # last 3 blocks
    for i in range(unfreeze_from, len(model.features)):
        for param in model.features[i].parameters():
            param.requires_grad = True
    
    # Replace classifier — always trainable
    model.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(1280, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params:     {total:,}")
    print(f"Trainable params: {trainable:,} ({100*trainable/total:.1f}%)")
    
    return model


print("Building MobileNetV2 (fine-tuning mode)...")
ft_model = build_mobilenetv2_finetune(num_classes=5)

# Two-phase training for fine-tuning:
# Phase 1: train only the head with LR=0.001
# Phase 2: fine-tune all unfrozen layers with very low LR=0.0001
print("\nPhase 1: Training head only (5 epochs)...")
ft_model = ft_model.to(device)
optimizer_phase1 = optim.Adam(filter(lambda p: p.requires_grad, ft_model.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    ft_model.train()
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer_phase1.zero_grad()
        criterion(ft_model(inputs), targets).backward()
        optimizer_phase1.step()

print("Phase 2: Fine-tuning last blocks + head (15 epochs, low LR)...")

# Use parameter groups: lower LR for pretrained layers
optimizer_phase2 = optim.Adam([
    {'params': ft_model.features.parameters(), 'lr': 1e-5},  # 10x lower for pretrained
    {'params': ft_model.classifier.parameters(), 'lr': 1e-4}
])

ft_train_accs, ft_val_accs = [], []
for epoch in range(15):
    ft_model.train()
    correct, total = 0, 0
    for inputs, targets in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer_phase2.zero_grad()
        out = ft_model(inputs)
        loss = criterion(out, targets)
        loss.backward()
        optimizer_phase2.step()
        _, pred = out.max(1)
        correct += pred.eq(targets).sum().item()
        total += targets.size(0)
    ft_train_accs.append(100. * correct / total)
    
    ft_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            _, pred = ft_model(inputs).max(1)
            correct += pred.eq(targets).sum().item()
            total += targets.size(0)
    ft_val_accs.append(100. * correct / total)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/15: Train {ft_train_accs[-1]:.1f}% | Val {ft_val_accs[-1]:.1f}%")

print(f"\nFine-tuning final Val Accuracy: {ft_val_accs[-1]:.1f}%")

In [ ]:
# ============================================================
# SECTION 5: Compare All Three Approaches
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Pad arrays to same length for comparison
max_len = max(len(scratch_val_accs), len(fe_val_accs), len(ft_val_accs))

def pad(lst, length):
    return lst + [lst[-1]] * (length - len(lst))

scratch_v = pad(scratch_val_accs, max_len)
fe_v = pad(fe_val_accs, max_len)
ft_v = pad(ft_val_accs, max_len)

x = range(1, max_len + 1)

axes[0].plot(x, scratch_v, label='From Scratch', color='#F44336', linewidth=2, linestyle='--')
axes[0].plot(x, fe_v, label='Feature Extraction (MobileNetV2)', color='#2196F3', linewidth=2)
axes[0].plot(x, ft_v, label='Fine-tuning (MobileNetV2)', color='#4CAF50', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[0].set_title('Transfer Learning vs Training From Scratch\n(1,000 training images)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 105)

# Bar chart of final results
methods = ['From Scratch', 'Feature\nExtraction', 'Fine-tuning']
final_accs = [scratch_val_accs[-1], fe_val_accs[-1], ft_val_accs[-1]]
colors = ['#F44336', '#2196F3', '#4CAF50']

bars = axes[1].bar(methods, final_accs, color=colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, acc in zip(bars, final_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

axes[1].set_ylabel('Final Validation Accuracy (%)', fontsize=12)
axes[1].set_title('Final Accuracy Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("RESULTS SUMMARY")
print("=" * 50)
for method, acc in zip(methods, final_accs):
    print(f"  {method.replace(chr(10), ' '):25s}: {acc:.1f}%")

improvement = ft_val_accs[-1] - scratch_val_accs[-1]
print(f"\nFine-tuning improvement over scratch: +{improvement:.1f}%")
print(f"Same dataset. Same number of epochs. Different starting point.")
print(f"This gap grows as dataset size decreases.")

In [ ]:
# ============================================================
# SECTION 6: Visualise Which Layers Are Frozen vs Trainable
# ============================================================

model_check = models.mobilenet_v2(pretrained=False)
for param in model_check.parameters():
    param.requires_grad = False

# Unfreeze last 3 blocks
for i in range(len(model_check.features) - 3, len(model_check.features)):
    for param in model_check.features[i].parameters():
        param.requires_grad = True

model_check.classifier = nn.Linear(1280, 5)

layers, frozen_counts, trainable_counts = [], [], []
for name, module in model_check.named_children():
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    frozen = total - trainable
    if total > 0:
        layers.append(name)
        frozen_counts.append(frozen)
        trainable_counts.append(trainable)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(layers))
bars1 = ax.bar(x, frozen_counts, label='Frozen (pretrained)', color='#78909C', alpha=0.8)
bars2 = ax.bar(x, trainable_counts, bottom=frozen_counts, label='Trainable', color='#4CAF50', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(layers, rotation=30, ha='right', fontsize=11)
ax.set_ylabel('Parameter Count', fontsize=12)
ax.set_title('MobileNetV2: Frozen vs Trainable Parameters per Layer Block', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

total_all = sum(frozen_counts) + sum(trainable_counts)
pct_frozen = 100 * sum(frozen_counts) / total_all
print(f"Total parameters:     {total_all:,}")
print(f"Frozen:               {sum(frozen_counts):,} ({pct_frozen:.1f}%)")
print(f"Trainable:            {sum(trainable_counts):,} ({100-pct_frozen:.1f}%)")
print(f"\nYou're getting ImageNet's representation power for only {100-pct_frozen:.0f}% of the compute.")

## Real World Problem: Domain Shift in Transfer Learning

A startup builds a plant disease classifier by fine-tuning ResNet50 pretrained on ImageNet. They get 91% accuracy on their validation set. They deploy it. Performance drops to 67% in production.

**What happened:** Domain shift. Their validation images were taken in a controlled lab setting under uniform lighting. In production, farmers take photos in the field under different lighting conditions, from different angles, with soil and other plants in the background. The distribution of production images is different from the training distribution.

**Why ImageNet pretraining didn't fully protect them:** ImageNet features are general enough to handle natural images, but fine-tuning on a narrow lab dataset can cause the model to overfit to lab-specific artefacts (uniform lighting, clean backgrounds) rather than the disease features themselves.

**The fix engineers actually use:**

1. **Domain-specific augmentation at training time:** Add random lighting changes, rotation, zoom, background variations that match what production photos actually look like.

2. **Test-time augmentation (TTA):** At inference, predict on multiple augmented versions of the same image and average the predictions. Increases robustness.

3. **Domain adaptation:** If you have some unlabelled production images, use techniques like CycleGAN or CORAL to align the training and production distributions.

4. **Monitor distribution shift in production:** Log input image statistics (brightness, contrast, pixel mean) over time. A shift in these distributions is an early signal that your model will start degrading.

## Interview Corner: MNC-Level Questions

---

**Q1: When would you choose feature extraction over fine-tuning?**

*What they're testing:* Whether you understand the data-size vs strategy relationship.

*Answer direction:* Feature extraction when: (a) your dataset is very small (<1,000 images) because fine-tuning would overfit the pretrained layers on so little data, (b) your task domain is close to the pretrained domain (e.g., natural image classification pretrained on ImageNet), (c) compute is limited — only training the head is much faster. Fine-tuning when: (a) you have more data (>5,000 images), (b) your domain is different from the pretraining data (medical images, satellite imagery), (c) you need the highest possible accuracy.

---

**Q2: Why must you use the same normalisation in transfer learning as was used during pretraining?**

*What they're testing:* Attention to preprocessing detail.

*Answer direction:* The pretrained weights learned to process inputs in a specific normalised range. If you feed in differently normalised inputs, the feature activations the pretrained layers produce will be on a completely different scale than what they were designed for. The pretrained layers won't behave as intended. For ImageNet pretrained models, you must use mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225] — not because these are magic numbers, but because these are the statistics of the ImageNet training set that the pretrained weights were calibrated against.

---

**Q3: You fine-tune a model and it performs worse than the pretrained baseline. What happened?**

*What they're testing:* Debugging fine-tuning failures.

*Answer direction:* Catastrophic forgetting. The fine-tuning overwrote the pretrained representations with noise because: (a) learning rate was too high — the model aggressively overwrites pretrained weights, (b) too many layers were unfrozen on too little data, (c) training ran too many epochs. Fix: use differential learning rates (10x lower LR for pretrained layers vs the head), start by training only the head for a few epochs before unfreezing anything, and use early stopping on validation loss.

---

**Q4: How would you choose between MobileNetV2, ResNet50, and EfficientNet-B4 for a production deployment?**

*What they're testing:* Practical model selection judgment.

*Answer direction:* Three constraints drive this: accuracy target, latency budget, and deployment environment. MobileNetV2: smallest, fastest, designed for mobile/edge (3.4M params, <100ms on CPU). Choose this for on-device inference. ResNet50: solid accuracy, moderate size (25M params). Good baseline for server inference where latency isn't critical. EfficientNet-B4: best accuracy per parameter, but heavier than MobileNet. Choose when accuracy is the primary constraint and you have GPU inference. In production I'd run all three against a validation set, measure latency on the actual deployment hardware, and pick based on which clears the accuracy threshold at the required latency.

---

**Q5: A company asks you to build an image classifier for a medical imaging task with 800 labelled images. Walk me through your approach.**

*What they're testing:* End-to-end thinking for low-data scenarios.

*Answer direction:* Start with a pretrained model — EfficientNet-B0 or ResNet50. Freeze all layers, train only the head for 10 epochs with aggressive data augmentation (flips, rotations, elastic deformations which are realistic for medical images). Evaluate. If underfitting, unfreeze the last 2 blocks and fine-tune at LR=1e-5. Use stratified k-fold cross-validation because 800 images is too small for a reliable single validation split. Optimise recall over precision — in medical imaging, false negatives (missing a disease) are worse than false positives. Use test-time augmentation at inference for additional robustness. Document everything for regulatory compliance.

## ML Spotlight

**Hugging Face `timm` (PyTorch Image Models)**

`timm` is the most comprehensive library for pretrained vision models in production. It provides 400+ pretrained models — EfficientNet, ResNet, ViT, ConvNeXt, Swin Transformer — all with consistent APIs, pretrained weights, and benchmark accuracy numbers.

It's the standard tool for transfer learning in production computer vision teams.

```python
import timm

# Load a pretrained model, swap the head for your task
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5)

# List all available models
model_names = timm.list_models('efficientnet*')
print(model_names)  # 30+ EfficientNet variants
```

GitHub: https://github.com/huggingface/pytorch-image-models

Benchmark results: https://github.com/huggingface/pytorch-image-models/blob/main/results

## Practice Exercise

1. Replace MobileNetV2 with ResNet18 (also available in `torchvision.models`). Apply the same feature extraction strategy. Compare accuracy.

2. Try progressive unfreezing: train head for 5 epochs → unfreeze last block, train 5 more epochs → unfreeze last 2 blocks, train 5 more. Plot accuracy at each stage.

3. Increase SAMPLES_PER_CLASS_TRAIN from 200 to 1000. Does the gap between feature extraction and fine-tuning change? What does this tell you about data size and strategy?

---

**What's Next**

Day 29: RNNs and LSTMs — How neural networks handle sequences, why vanilla RNNs fail on long sequences, and how LSTM gates solve the vanishing gradient problem.